# Transfer Learning: CNN Architecture Comparison

**Classroom version — one architecture at a time**

In this notebook, we compare six ImageNet-pretrained CNN architectures on CIFAR-10:

1. VGG16
2. InceptionV3
3. ResNet50
4. DenseNet121
5. MobileNetV2
6. EfficientNetB0

For every architecture, we follow the same visible workflow:

**Create the pretrained base → freeze its layers → add a classifier → compile → train → evaluate → measure model size and prediction speed.**

The code for each architecture is written separately. There is no dictionary of model builders and no loop that trains all architectures automatically. This makes it easier to teach and discuss each model independently.

> This is a feature-extraction experiment: the pretrained CNN is frozen, and only the newly added classification head is trained.

## 1. Import libraries

In [ ]:
import os
import time
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Set experiment parameters
These settings are shared by all six experiments. For a quick classroom demonstration, reduce `TRAIN_LIMIT`, `TEST_LIMIT`, or `EPOCHS`. Use `None` to use the complete dataset.

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 3

TRAIN_LIMIT = None   # Example: 5000; None means all training images
TEST_LIMIT = None    # Example: 1000; None means all test images

NUM_CLASSES = 10
RESULTS_DIR = "cnn_comparison_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

tf.random.set_seed(42)
np.random.seed(42)

## 3. Load and explore CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.flatten()
y_test = y_test.flatten()

if TRAIN_LIMIT is not None:
    x_train, y_train = x_train[:TRAIN_LIMIT], y_train[:TRAIN_LIMIT]

if TEST_LIMIT is not None:
    x_test, y_test = x_test[:TEST_LIMIT], y_test[:TEST_LIMIT]

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training images:", x_train.shape)
print("Test images:", x_test.shape)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Prepare the TensorFlow datasets

CIFAR-10 images are originally 32×32 pixels. Transfer-learning architectures expect larger images, so we resize them to 160×160.

The labels are converted to one-hot vectors because we use `categorical_crossentropy` as the loss function.

The architecture-specific `preprocess_input` function will be applied inside each model.

In [ ]:
def make_dataset(images, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))

    if training:
        dataset = dataset.shuffle(5000)

    dataset = dataset.map(
        lambda image, label: (
            tf.image.resize(tf.cast(image, tf.float32), (IMG_SIZE, IMG_SIZE)),
            tf.one_hot(label, NUM_CLASSES)
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(x_train, y_train, training=True)
test_ds = make_dataset(x_test, y_test)

print("Datasets are ready.")

## 5. VGG16

**Architecture workflow:** load ImageNet weights, remove the original classifier, freeze the convolutional base, and attach a new CIFAR-10 classifier.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

vgg16_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

vgg16_base.trainable = False

vgg16_inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Lambda(preprocess_input)(vgg16_inputs)
x = vgg16_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
vgg16_outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

vgg16_model = keras.Model(vgg16_inputs, vgg16_outputs, name="VGG16_CIFAR10")

vgg16_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

vgg16_model.summary()

In [ ]:
vgg16_start = time.perf_counter()

vgg16_history = vgg16_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

vgg16_training_seconds = time.perf_counter() - vgg16_start

In [ ]:
vgg16_test_loss, vgg16_test_accuracy = vgg16_model.evaluate(test_ds, verbose=0)

vgg16_parameter_count = vgg16_model.count_params()
vgg16_path = os.path.join(RESULTS_DIR, "VGG16.keras")
vgg16_model.save(vgg16_path)
vgg16_size_mb = os.path.getsize(vgg16_path) / (1024 * 1024)

vgg16_sample = next(iter(test_ds))[0][:BATCH_SIZE]
vgg16_model.predict(vgg16_sample, verbose=0)  # warm-up

vgg16_start = time.perf_counter()
vgg16_model.predict(vgg16_sample, verbose=0)
vgg16_prediction_ms = (
    time.perf_counter() - vgg16_start
) / int(vgg16_sample.shape[0]) * 1000

print("VGG16 test loss:", round(vgg16_test_loss, 4))
print("VGG16 test accuracy:", f"{vgg16_test_accuracy:.2%}")
print("Training time (seconds):", round(vgg16_training_seconds, 2))
print("Parameters:", f"{vgg16_parameter_count:,}")
print("Saved model size (MB):", round(vgg16_size_mb, 2))
print("Prediction speed (ms/image):", round(vgg16_prediction_ms, 3))

## 6. InceptionV3

InceptionV3 uses Inception modules to process information at multiple filter scales. We repeat the same transfer-learning workflow, but use InceptionV3's own preprocessing function.

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

inception_base = InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
inception_base.trainable = False

inception_inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Lambda(inception_preprocess)(inception_inputs)
x = inception_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
inception_outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

inception_model = keras.Model(inception_inputs, inception_outputs, name="InceptionV3_CIFAR10")
inception_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
inception_model.summary()

In [ ]:
inception_start = time.perf_counter()
inception_history = inception_model.fit(
    train_ds, validation_data=test_ds, epochs=EPOCHS
)
inception_training_seconds = time.perf_counter() - inception_start

In [ ]:
inception_test_loss, inception_test_accuracy = inception_model.evaluate(test_ds, verbose=0)
inception_parameter_count = inception_model.count_params()

inception_path = os.path.join(RESULTS_DIR, "InceptionV3.keras")
inception_model.save(inception_path)
inception_size_mb = os.path.getsize(inception_path) / (1024 * 1024)

inception_sample = next(iter(test_ds))[0][:BATCH_SIZE]
inception_model.predict(inception_sample, verbose=0)
inception_start = time.perf_counter()
inception_model.predict(inception_sample, verbose=0)
inception_prediction_ms = (
    time.perf_counter() - inception_start
) / int(inception_sample.shape[0]) * 1000

print("InceptionV3 test loss:", round(inception_test_loss, 4))
print("InceptionV3 test accuracy:", f"{inception_test_accuracy:.2%}")
print("Training time (seconds):", round(inception_training_seconds, 2))
print("Parameters:", f"{inception_parameter_count:,}")
print("Saved model size (MB):", round(inception_size_mb, 2))
print("Prediction speed (ms/image):", round(inception_prediction_ms, 3))

## 7. ResNet50

ResNet50 uses residual connections (skip connections), which help information and gradients flow through a deep network.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
resnet_base.trainable = False

resnet_inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Lambda(resnet_preprocess)(resnet_inputs)
x = resnet_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
resnet_outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

resnet_model = keras.Model(resnet_inputs, resnet_outputs, name="ResNet50_CIFAR10")
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
resnet_model.summary()

In [ ]:
resnet_start = time.perf_counter()
resnet_history = resnet_model.fit(
    train_ds, validation_data=test_ds, epochs=EPOCHS
)
resnet_training_seconds = time.perf_counter() - resnet_start

In [ ]:
resnet_test_loss, resnet_test_accuracy = resnet_model.evaluate(test_ds, verbose=0)
resnet_parameter_count = resnet_model.count_params()

resnet_path = os.path.join(RESULTS_DIR, "ResNet50.keras")
resnet_model.save(resnet_path)
resnet_size_mb = os.path.getsize(resnet_path) / (1024 * 1024)

resnet_sample = next(iter(test_ds))[0][:BATCH_SIZE]
resnet_model.predict(resnet_sample, verbose=0)
resnet_start = time.perf_counter()
resnet_model.predict(resnet_sample, verbose=0)
resnet_prediction_ms = (
    time.perf_counter() - resnet_start
) / int(resnet_sample.shape[0]) * 1000

print("ResNet50 test loss:", round(resnet_test_loss, 4))
print("ResNet50 test accuracy:", f"{resnet_test_accuracy:.2%}")
print("Training time (seconds):", round(resnet_training_seconds, 2))
print("Parameters:", f"{resnet_parameter_count:,}")
print("Saved model size (MB):", round(resnet_size_mb, 2))
print("Prediction speed (ms/image):", round(resnet_prediction_ms, 3))

## 8. DenseNet121

DenseNet connects each layer to later layers through feature concatenation. This encourages feature reuse.

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

densenet_base = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
densenet_base.trainable = False

densenet_inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Lambda(densenet_preprocess)(densenet_inputs)
x = densenet_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
densenet_outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

densenet_model = keras.Model(densenet_inputs, densenet_outputs, name="DenseNet121_CIFAR10")
densenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
densenet_model.summary()

In [ ]:
densenet_start = time.perf_counter()
densenet_history = densenet_model.fit(
    train_ds, validation_data=test_ds, epochs=EPOCHS
)
densenet_training_seconds = time.perf_counter() - densenet_start

In [ ]:
densenet_test_loss, densenet_test_accuracy = densenet_model.evaluate(test_ds, verbose=0)
densenet_parameter_count = densenet_model.count_params()

densenet_path = os.path.join(RESULTS_DIR, "DenseNet121.keras")
densenet_model.save(densenet_path)
densenet_size_mb = os.path.getsize(densenet_path) / (1024 * 1024)

densenet_sample = next(iter(test_ds))[0][:BATCH_SIZE]
densenet_model.predict(densenet_sample, verbose=0)
densenet_start = time.perf_counter()
densenet_model.predict(densenet_sample, verbose=0)
densenet_prediction_ms = (
    time.perf_counter() - densenet_start
) / int(densenet_sample.shape[0]) * 1000

print("DenseNet121 test loss:", round(densenet_test_loss, 4))
print("DenseNet121 test accuracy:", f"{densenet_test_accuracy:.2%}")
print("Training time (seconds):", round(densenet_training_seconds, 2))
print("Parameters:", f"{densenet_parameter_count:,}")
print("Saved model size (MB):", round(densenet_size_mb, 2))
print("Prediction speed (ms/image):", round(densenet_prediction_ms, 3))

## 9. MobileNetV2

MobileNetV2 is designed for efficient computation, making it useful for mobile and resource-constrained devices.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

mobilenet_base = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
mobilenet_base.trainable = False

mobilenet_inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Lambda(mobilenet_preprocess)(mobilenet_inputs)
x = mobilenet_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
mobilenet_outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

mobilenet_model = keras.Model(mobilenet_inputs, mobilenet_outputs, name="MobileNetV2_CIFAR10")
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
mobilenet_model.summary()

In [ ]:
mobilenet_start = time.perf_counter()
mobilenet_history = mobilenet_model.fit(
    train_ds, validation_data=test_ds, epochs=EPOCHS
)
mobilenet_training_seconds = time.perf_counter() - mobilenet_start

In [ ]:
mobilenet_test_loss, mobilenet_test_accuracy = mobilenet_model.evaluate(test_ds, verbose=0)
mobilenet_parameter_count = mobilenet_model.count_params()

mobilenet_path = os.path.join(RESULTS_DIR, "MobileNetV2.keras")
mobilenet_model.save(mobilenet_path)
mobilenet_size_mb = os.path.getsize(mobilenet_path) / (1024 * 1024)

mobilenet_sample = next(iter(test_ds))[0][:BATCH_SIZE]
mobilenet_model.predict(mobilenet_sample, verbose=0)
mobilenet_start = time.perf_counter()
mobilenet_model.predict(mobilenet_sample, verbose=0)
mobilenet_prediction_ms = (
    time.perf_counter() - mobilenet_start
) / int(mobilenet_sample.shape[0]) * 1000

print("MobileNetV2 test loss:", round(mobilenet_test_loss, 4))
print("MobileNetV2 test accuracy:", f"{mobilenet_test_accuracy:.2%}")
print("Training time (seconds):", round(mobilenet_training_seconds, 2))
print("Parameters:", f"{mobilenet_parameter_count:,}")
print("Saved model size (MB):", round(mobilenet_size_mb, 2))
print("Prediction speed (ms/image):", round(mobilenet_prediction_ms, 3))

## 10. EfficientNetB0

EfficientNet scales network depth, width, and input resolution in a coordinated way. Keras EfficientNetB0 includes its input rescaling, so its `preprocess_input` function is effectively a pass-through for this application.

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

efficientnet_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
efficientnet_base.trainable = False

efficientnet_inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Lambda(efficientnet_preprocess)(efficientnet_inputs)
x = efficientnet_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
efficientnet_outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

efficientnet_model = keras.Model(
    efficientnet_inputs, efficientnet_outputs, name="EfficientNetB0_CIFAR10"
)
efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
efficientnet_model.summary()

In [ ]:
efficientnet_start = time.perf_counter()
efficientnet_history = efficientnet_model.fit(
    train_ds, validation_data=test_ds, epochs=EPOCHS
)
efficientnet_training_seconds = time.perf_counter() - efficientnet_start

In [ ]:
efficientnet_test_loss, efficientnet_test_accuracy = efficientnet_model.evaluate(test_ds, verbose=0)
efficientnet_parameter_count = efficientnet_model.count_params()

efficientnet_path = os.path.join(RESULTS_DIR, "EfficientNetB0.keras")
efficientnet_model.save(efficientnet_path)
efficientnet_size_mb = os.path.getsize(efficientnet_path) / (1024 * 1024)

efficientnet_sample = next(iter(test_ds))[0][:BATCH_SIZE]
efficientnet_model.predict(efficientnet_sample, verbose=0)
efficientnet_start = time.perf_counter()
efficientnet_model.predict(efficientnet_sample, verbose=0)
efficientnet_prediction_ms = (
    time.perf_counter() - efficientnet_start
) / int(efficientnet_sample.shape[0]) * 1000

print("EfficientNetB0 test loss:", round(efficientnet_test_loss, 4))
print("EfficientNetB0 test accuracy:", f"{efficientnet_test_accuracy:.2%}")
print("Training time (seconds):", round(efficientnet_training_seconds, 2))
print("Parameters:", f"{efficientnet_parameter_count:,}")
print("Saved model size (MB):", round(efficientnet_size_mb, 2))
print("Prediction speed (ms/image):", round(efficientnet_prediction_ms, 3))

## 11. Build the comparison table

At this point, each architecture has been trained and evaluated in its own section. We now collect the six sets of measurements into one table.

In [ ]:
results_df = pd.DataFrame([
    {
        "Architecture": "VGG16",
        "Accuracy": vgg16_test_accuracy,
        "Training Time (seconds)": vgg16_training_seconds,
        "Parameters": vgg16_parameter_count,
        "Model Size (MB)": vgg16_size_mb,
        "Prediction Speed (ms/image)": vgg16_prediction_ms
    },
    {
        "Architecture": "InceptionV3",
        "Accuracy": inception_test_accuracy,
        "Training Time (seconds)": inception_training_seconds,
        "Parameters": inception_parameter_count,
        "Model Size (MB)": inception_size_mb,
        "Prediction Speed (ms/image)": inception_prediction_ms
    },
    {
        "Architecture": "ResNet50",
        "Accuracy": resnet_test_accuracy,
        "Training Time (seconds)": resnet_training_seconds,
        "Parameters": resnet_parameter_count,
        "Model Size (MB)": resnet_size_mb,
        "Prediction Speed (ms/image)": resnet_prediction_ms
    },
    {
        "Architecture": "DenseNet121",
        "Accuracy": densenet_test_accuracy,
        "Training Time (seconds)": densenet_training_seconds,
        "Parameters": densenet_parameter_count,
        "Model Size (MB)": densenet_size_mb,
        "Prediction Speed (ms/image)": densenet_prediction_ms
    },
    {
        "Architecture": "MobileNetV2",
        "Accuracy": mobilenet_test_accuracy,
        "Training Time (seconds)": mobilenet_training_seconds,
        "Parameters": mobilenet_parameter_count,
        "Model Size (MB)": mobilenet_size_mb,
        "Prediction Speed (ms/image)": mobilenet_prediction_ms
    },
    {
        "Architecture": "EfficientNetB0",
        "Accuracy": efficientnet_test_accuracy,
        "Training Time (seconds)": efficientnet_training_seconds,
        "Parameters": efficientnet_parameter_count,
        "Model Size (MB)": efficientnet_size_mb,
        "Prediction Speed (ms/image)": efficientnet_prediction_ms
    }
])

display(results_df.style.format({
    "Accuracy": "{:.2%}",
    "Training Time (seconds)": "{:.2f}",
    "Parameters": "{:,.0f}",
    "Model Size (MB)": "{:.2f}",
    "Prediction Speed (ms/image)": "{:.3f}"
}))

results_df.to_csv(
    os.path.join(RESULTS_DIR, "comparison_results.csv"),
    index=False
)

## 12. Visualize the comparison
Each metric is plotted separately because the metrics use different units and scales.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["Architecture"], results_df["Accuracy"] * 100)
plt.title("Test Accuracy Comparison")
plt.xlabel("CNN Architecture")
plt.ylabel("Accuracy (%)")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["Architecture"], results_df["Training Time (seconds)"])
plt.title("Training Time Comparison")
plt.xlabel("CNN Architecture")
plt.ylabel("Training time (seconds)")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["Architecture"], results_df["Parameters"])
plt.title("Parameter Count Comparison")
plt.xlabel("CNN Architecture")
plt.ylabel("Number of parameters")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["Architecture"], results_df["Model Size (MB)"])
plt.title("Saved Model Size Comparison")
plt.xlabel("CNN Architecture")
plt.ylabel("Model size (MB)")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["Architecture"], results_df["Prediction Speed (ms/image)"])
plt.title("Prediction Speed Comparison")
plt.xlabel("CNN Architecture")
plt.ylabel("Milliseconds per image")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Predict the class of an example image

Use one of the trained models to classify a test image. Change `selected_model` to another model variable if you want to demonstrate a different architecture.

In [ ]:
selected_model = mobilenet_model
image_index = 0

image = x_test[image_index]
resized_image = tf.image.resize(
    tf.cast(image, tf.float32),
    (IMG_SIZE, IMG_SIZE)
)
prediction = selected_model.predict(
    tf.expand_dims(resized_image, axis=0),
    verbose=0
)

predicted_class = np.argmax(prediction[0])

plt.imshow(image)
plt.title(
    f"Actual: {class_names[y_test[image_index]]} | "
    f"Predicted: {class_names[predicted_class]}"
)
plt.axis("off")
plt.show()

print("Predicted probabilities:", prediction[0])

## 14. Class discussion questions

1. Which architecture achieved the highest test accuracy?
2. Which architecture required the least training time?
3. Which model has the largest number of parameters?
4. Does a larger model always achieve higher accuracy?
5. Which model has the smallest saved file size?
6. Which model has the fastest prediction speed on this computer?
7. Why is MobileNetV2 suitable for mobile or resource-constrained devices?
8. What is the difference between feature extraction and fine-tuning?
9. Why do we use a different `preprocess_input` function for different architectures?

## Important notes

- All six models use ImageNet-pretrained weights and have their base layers frozen.
- The new classification head is trained on CIFAR-10.
- The first run downloads ImageNet weights and requires internet access.
- The experiment can take a long time, especially on CPU. For a classroom demonstration, use smaller `TRAIN_LIMIT` and `TEST_LIMIT` values.
- Results depend on hardware, TensorFlow version, image size, batch size, and number of epochs.
- Prediction speed is a simple batch-based measurement on the current machine. Compare models under the same conditions.
- The saved `.keras` file size is measured after saving each model.
- The test set is used for evaluation here for a teaching comparison. For a rigorous experiment, create a separate validation set for model selection and keep the test set for final evaluation.
- The six models are kept in memory so that the final prediction example can use them. If memory is limited, restart the runtime after recording a model's results, or run the architectures in separate sessions.